<a href="https://colab.research.google.com/github/ChristepherCBiju/Addon_project/blob/main/project5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
import joblib

In [ ]:
def train_and_save_model():

    # Load dataset
    iris = load_iris()
    X, y = iris.data, iris.target

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Model training
    model = DecisionTreeClassifier()
    model.fit(X_train, y_train)

    # Save model
    joblib.dump(model, "decision_tree_model.pkl")
    print("Model training complete and saved as decision_tree_model.pkl")

In [ ]:
if __name__ == "__main__":
    train_and_save_model()

Model training complete and saved as decision_tree_model.pkl


In [ ]:
%%writefile main.py
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np

# Load trained model
model = joblib.load("decision_tree_model.pkl")

app = FastAPI(title="Decision Tree Classifier API")

class IrisFeatures(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float

@app.get("/")
def home():
    return {"message": "FastAPI Decision Tree Model Running!"}

@app.post("/predict")
def predict(data: IrisFeatures):
    arr = np.array([[
        data.sepal_length,
        data.sepal_width,
        data.petal_length,
        data.petal_width
    ]])

    pred = model.predict(arr)[0]
    classes = ["setosa", "versicolor", "virginica"]
    return {"prediction": classes[pred]}

Overwriting main.py


In [ ]:
#pip install pyngrok

In [ ]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "36Q58evTQodpCbDbjogdmEfmlI4_WxYR4gmhBQtJNqKqA5es"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [ ]:
import nest_asyncio
import uvicorn
from pyngrok import ngrok
import asyncio # Import asyncio

# Apply patch
nest_asyncio.apply()

# Kill running tunnels (if any)
ngrok.kill()

# Start ngrok tunnel
public_url = ngrok.connect(8000)
print("Public URL:", public_url)

# Start API
# Explicitly create a Uvicorn Server instance to run within the existing event loop
config = uvicorn.Config("main:app", host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)

# Run the server's serve coroutine on the existing event loop
try:
    asyncio.get_event_loop().run_until_complete(server.serve())
except KeyboardInterrupt:
    print("Server stopped.")

Public URL: NgrokTunnel: "https://carbamic-cottony-izola.ngrok-free.dev" -> "http://localhost:8000"


INFO:     Started server process [389]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-1' coro=<Server.serve() done, defined at /usr/local/lib/python3.12/dist-packages/uvicorn/server.py:69> exception=KeyboardInterrupt()>
Traceback (most recent call last):
  File "/tmp/ipython-input-1336418803.py", line 23, in <cell line: 0>
    asyncio.get_event_loop().run_until_complete(server.serve())
  File "/usr/local/lib/python3.12/dist-packages/nest_asyncio.py", line 92, in run_until_complete
    self._run_once()
  File "/usr/local/lib/python3.12/dist-packages/nest_asyncio.py", line 133, in _run_once
    handle._run()
  File "/usr/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
  File "/usr/lib/python3.12/asyncio/tasks.py", line 396, in __wakeup
    self._

INFO:     61.2.64.4:0 - "GET / HTTP/1.1" 200 OK
INFO:     61.2.64.4:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     61.2.64.4:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     61.2.64.4:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     61.2.64.4:0 - "POST /predict HTTP/1.1" 200 OK
